In [1]:
import sys
from pathlib import Path
sys.path.append(str(Path().resolve().parents[0]))

import numpy as np
import pandas as pd

from config import PROCESSED_DATA_DIR

In [2]:
modeling_df = pd.read_parquet(PROCESSED_DATA_DIR / "modeling_dataset.parquet")
predictor_cols = pd.read_csv(PROCESSED_DATA_DIR / "modeling_predictor_columns.csv")["predictor_column"].tolist()

print(modeling_df.shape)
print(f"Initial predictor columns: {len(predictor_cols)}")
modeling_df.head()

(94444, 457)
Initial predictor columns: 448


,subject_id,hadm_id,stay_id,first_careunit,intime,outtime,los,gender,anchor_age,admittime,...,other_medication_mcg_total_24h,other_medication_meq_total_24h,other_medication_mg_total_24h,other_ml_total_24h,other_ounces_total_24h,sedative_analgesic_mcg_total_24h,sedative_analgesic_mg_total_24h,vasopressor_inotrope_mcg_total_24h,vasopressor_inotrope_mg_total_24h,vasopressor_inotrope_units_total_24h
0,10000032,29079034,39553978,Medical Intensive Care Unit (MICU),2180-07-23 14:00:00,2180-07-23 23:50:47,0.410266,F,52,2180-07-23 12:35:00,...,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.00000,0.0
1,10000690,25860671,37081114,Medical Intensive Care Unit (MICU),2150-11-02 19:37:00,2150-11-06 17:03:17,3.893252,F,86,2150-11-02 18:02:00,...,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,9.34123,0.0
2,10000980,26913865,39765666,Medical Intensive Care Unit (MICU),2189-06-27 08:42:00,2189-06-27 20:38:27,0.497535,F,73,2189-06-27 07:38:00,...,0.0,0.0,40.000001,0.0,0.0,0.0,0.0,0.0,0.00000,0.0
3,10001217,24597018,37067082,Surgical Intensive Care Unit (SICU),2157-11-20 19:18:02,2157-11-21 22:08:00,1.118032,F,55,2157-11-18 22:56:00,...,0.0,0.0,10.000000,0.0,0.0,0.0,2.0,0.0,0.00000,0.0
4,10001217,27703517,34592300,Surgical Intensive Care Unit (SICU),2157-12-19 15:42:24,2157-12-20 14:27:41,0.948113,F,55,2157-12-18 16:58:00,...,0.0,0.0,1009.999953,0.0,0.0,0.0,6.0,0.0,0.00000,0.0


In [3]:
required_cols = ["subject_id", "stay_id", "duration", "event_observed"]
missing_required_cols = [col for col in required_cols if col not in modeling_df.columns]

if missing_required_cols:
    raise ValueError(f"Missing required columns: {missing_required_cols}")

if modeling_df["stay_id"].duplicated().any():
    raise ValueError("Dataset has duplicate stay_id rows")

if modeling_df["duration"].isna().any():
    raise ValueError("duration has missing values")

if (modeling_df["duration"] <= 0).any():
    raise ValueError("duration has non-positive values")

if not modeling_df["event_observed"].isin([0, 1]).all():
    raise ValueError("event_observed must be binary")

print("Core dataset validation passed")

Core dataset validation passed


In [4]:
outcome_summary = modeling_df["duration"].describe(percentiles=[0.01, 0.05, 0.25, 0.5, 0.75, 0.95, 0.99])
event_summary = modeling_df["event_observed"].value_counts(dropna=False).rename("count").to_frame()
event_summary["pct"] = event_summary["count"] / len(modeling_df)

print(outcome_summary)
print()
print("Event observed distribution:")
print(event_summary)

if modeling_df["event_observed"].nunique() == 1:
    print()
    print("Note: event_observed has one value only. This is a time-to-ICU-discharge setup with no censoring.")

count    94444.000000
mean         3.630025
std          5.402474
min          0.001250
1%           0.209321
5%           0.543570
25%          1.096212
50%          1.965648
75%          3.862575
95%         12.681538
99%         26.438372
max        226.403079
Name: duration, dtype: float64

Event observed distribution:
                count  pct
event_observed            
1               94444  1.0

Note: event_observed has one value only. This is a time-to-ICU-discharge setup with no censoring.


In [5]:
duration_quantiles = modeling_df["duration"].quantile([0.5, 0.75, 0.9, 0.95, 0.99, 0.995, 1.0])
long_stay_threshold = modeling_df["duration"].quantile(0.99)
long_stay_count = modeling_df["duration"].gt(long_stay_threshold).sum()

print(duration_quantiles)
print()
print(f"Rows above 99th percentile duration ({long_stay_threshold:.2f} days): {long_stay_count:,}")
print("Keep these rows for now; decide on truncation or robust modeling in the model-fitting notebook.")

0.500      1.965648
0.750      3.862575
0.900      7.920927
0.950     12.681538
0.990     26.438372
0.995     33.986896
1.000    226.403079
Name: duration, dtype: float64

Rows above 99th percentile duration (26.44 days): 945
Keep these rows for now; decide on truncation or robust modeling in the model-fitting notebook.


In [6]:
leakage_exact_cols = {
    "los",
    "duration",
    "event_observed",
    "outtime",
    "intime",
    "admittime",
    "subject_id",
    "hadm_id",
    "stay_id",
}

leakage_name_patterns = [
    "outtime",
    "dischtime",
    "deathtime",
    "event_observed",
    "los",
]

predictor_cols = [col for col in predictor_cols if col in modeling_df.columns]
leakage_cols = [
    col for col in predictor_cols
    if col in leakage_exact_cols or any(pattern in col.lower() for pattern in leakage_name_patterns)
]

if leakage_cols:
    raise ValueError(f"Potential leakage columns found in predictors: {leakage_cols}")

print(f"Leakage check passed for {len(predictor_cols)} predictors")

Leakage check passed for 448 predictors


In [7]:
X = modeling_df[predictor_cols].copy()

missing_summary_df = pd.DataFrame({
    "missing_count": X.isna().sum(),
    "missing_pct": X.isna().mean(),
    "dtype": X.dtypes.astype(str),
    "nunique_dropna": X.nunique(dropna=True),
}).sort_values("missing_pct", ascending=False)

missing_summary_df.head(30)

,missing_count,missing_pct,dtype,nunique_dropna
albumin_std_24h,89898,0.951866,float64,661
alkaline_phosphatase_std_24h,82319,0.871617,float64,2081
alt_std_24h,82116,0.869468,float64,2690
bilirubin_total_std_24h,82084,0.869129,float64,1912
ast_std_24h,82008,0.868324,float64,3194
albumin_trend_24h,69144,0.732116,float64,125
albumin_first_24h,69144,0.732116,float64,56
albumin_max_24h,69144,0.732116,float64,54
albumin_median_24h,69144,0.732116,float64,129
albumin_mean_24h,69144,0.732116,float64,334


In [8]:
high_missing_threshold = 0.80
high_missing_cols = missing_summary_df.index[missing_summary_df["missing_pct"].gt(high_missing_threshold)].tolist()

print(f"Columns with >{high_missing_threshold:.0%} missingness: {len(high_missing_cols)}")
print(high_missing_cols[:50])

Columns with >80% missingness: 5
['albumin_std_24h', 'alkaline_phosphatase_std_24h', 'alt_std_24h', 'bilirubin_total_std_24h', 'ast_std_24h']


In [9]:
low_variance_cols = missing_summary_df.index[missing_summary_df["nunique_dropna"].le(1)].tolist()

print(f"Low-variance/all-constant predictor columns: {len(low_variance_cols)}")
print(low_variance_cols[:80])

Low-variance/all-constant predictor columns: 0
[]


In [10]:
numeric_cols = X.select_dtypes(include=["number", "bool"]).columns.tolist()
categorical_cols = [col for col in X.columns if col not in numeric_cols]

print(f"Numeric predictors: {len(numeric_cols)}")
print(f"Categorical predictors: {len(categorical_cols)}")
print("Categorical predictors:", categorical_cols)

Numeric predictors: 436
Categorical predictors: 12
Categorical predictors: ['first_careunit', 'gender', 'admission_type', 'admission_location', 'insurance', 'marital_status', 'language_grouped', 'race_grouped', 'age_group', 'admission_urgency_group', 'admission_source_group', 'careunit_group']


In [11]:
categorical_summary = []

for col in categorical_cols:
    value_counts = X[col].value_counts(dropna=False)
    top_value = value_counts.index[0] if len(value_counts) else None
    top_count = value_counts.iloc[0] if len(value_counts) else 0
    categorical_summary.append({
        "column": col,
        "nunique_dropna": X[col].nunique(dropna=True),
        "missing_pct": X[col].isna().mean(),
        "top_value": top_value,
        "top_pct": top_count / len(X),
    })

categorical_summary_df = pd.DataFrame(categorical_summary).sort_values(
    ["nunique_dropna", "missing_pct"],
    ascending=[False, False]
)

categorical_summary_df

,column,nunique_dropna,missing_pct,top_value,top_pct
0,first_careunit,17,0.0,Medical Intensive Care Unit (MICU),0.219167
3,admission_location,11,0.0,EMERGENCY ROOM,0.397071
2,admission_type,9,0.0,EW EMER.,0.511933
4,insurance,6,0.0,Medicare,0.548632
6,language_grouped,6,0.0,English,0.900491
7,race_grouped,6,0.0,WHITE,0.663494
10,admission_source_group,6,0.0,ED,0.397071
5,marital_status,5,0.0,MARRIED,0.443660
11,careunit_group,5,0.0,MEDICAL,0.382724
8,age_group,4,0.0,60-79,0.446392


In [12]:
nonfinite_summary = []

for col in numeric_cols:
    values = X[col].to_numpy(dtype="float64", copy=False)
    nonfinite_count = (~np.isfinite(values) & ~pd.isna(values)).sum()
    if nonfinite_count > 0:
        nonfinite_summary.append({"column": col, "nonfinite_count": int(nonfinite_count)})

nonfinite_summary_df = pd.DataFrame(nonfinite_summary)

if not nonfinite_summary_df.empty:
    raise ValueError("Non-finite numeric values found")

print("Non-finite numeric check passed")

Non-finite numeric check passed


In [13]:
numeric_missing_pct = X[numeric_cols].isna().mean().sort_values(ascending=False)
print("Numeric columns with highest missingness:")
print(numeric_missing_pct.head(30))

complete_numeric_cols = numeric_missing_pct[numeric_missing_pct.eq(0)].index.tolist()
print()
print(f"Complete numeric predictors: {len(complete_numeric_cols)}")

Numeric columns with highest missingness:
albumin_std_24h                    0.951866
alkaline_phosphatase_std_24h       0.871617
alt_std_24h                        0.869468
bilirubin_total_std_24h            0.869129
ast_std_24h                        0.868324
albumin_last_24h                   0.732116
albumin_mean_24h                   0.732116
albumin_min_24h                    0.732116
albumin_first_24h                  0.732116
albumin_max_24h                    0.732116
albumin_median_24h                 0.732116
albumin_trend_24h                  0.732116
lactate_std_24h                    0.666893
pco2_std_24h                       0.616386
po2_std_24h                        0.616228
inr_std_24h                        0.611166
pt_std_24h                         0.610997
ph_std_24h                         0.602357
ptt_std_24h                        0.598132
bilirubin_total_max_24h            0.571280
bilirubin_total_trend_24h          0.571280
bilirubin_total_mean_24h          

In [14]:
correlation_sample_cols = [
    col for col in numeric_cols
    if col not in high_missing_cols and col not in low_variance_cols
]

corr_input_df = X[correlation_sample_cols].copy()
for col in corr_input_df.columns:
    if corr_input_df[col].isna().any():
        corr_input_df[col] = corr_input_df[col].fillna(corr_input_df[col].median())

corr_matrix = corr_input_df.corr().abs()
upper_mask = np.triu(np.ones(corr_matrix.shape), k=1).astype(bool)
corr_pairs_df = (
    corr_matrix.where(upper_mask)
    .stack()
    .rename("abs_correlation")
    .reset_index()
    .rename(columns={"level_0": "feature_1", "level_1": "feature_2"})
    .sort_values("abs_correlation", ascending=False)
)

high_corr_threshold = 0.98
high_corr_pairs_df = corr_pairs_df[corr_pairs_df["abs_correlation"].ge(high_corr_threshold)]

print(f"Feature pairs with abs(correlation) >= {high_corr_threshold}: {len(high_corr_pairs_df)}")
high_corr_pairs_df.head(30)

Feature pairs with abs(correlation) >= 0.98: 137


,feature_1,feature_2,abs_correlation
89673,other_event_count_24h,other_duration_hours_24h,1.000000
89106,antibiotic_event_count_24h,antibiotic_duration_hours_24h,0.999910
89595,oral_gastric_event_count_24h,oral_gastric_duration_hours_24h,0.999909
3415,dbp_count_24h,sbp_count_24h,0.999880
87915,pco2_observed_24h,po2_observed_24h,0.999787
56106,alkaline_phosphatase_mean_24h,alkaline_phosphatase_median_24h,0.999768
57446,bilirubin_total_mean_24h,bilirubin_total_median_24h,0.999765
36388,pco2_count_24h,po2_count_24h,0.999618
55835,albumin_mean_24h,albumin_median_24h,0.999535
35382,inr_count_24h,pt_count_24h,0.999497


In [15]:
drop_for_readiness = sorted(set(high_missing_cols + low_variance_cols))
ready_predictor_cols = [col for col in predictor_cols if col not in drop_for_readiness]

ready_numeric_cols = [col for col in numeric_cols if col in ready_predictor_cols]
ready_categorical_cols = [col for col in categorical_cols if col in ready_predictor_cols]

print(f"Original predictors: {len(predictor_cols)}")
print(f"Dropped for readiness: {len(drop_for_readiness)}")
print(f"Ready predictors: {len(ready_predictor_cols)}")
print(f"Ready numeric predictors: {len(ready_numeric_cols)}")
print(f"Ready categorical predictors: {len(ready_categorical_cols)}")

Original predictors: 448
Dropped for readiness: 5
Ready predictors: 443
Ready numeric predictors: 431
Ready categorical predictors: 12


In [16]:
rng = np.random.default_rng(42)
unique_subject_ids = modeling_df["subject_id"].drop_duplicates().to_numpy().copy()
rng.shuffle(unique_subject_ids)

test_fraction = 0.20
test_subject_count = int(len(unique_subject_ids) * test_fraction)
test_subject_ids = set(unique_subject_ids[:test_subject_count])

split_df = modeling_df[["subject_id", "stay_id", "duration", "event_observed"]].copy()
split_df["split"] = np.where(split_df["subject_id"].isin(test_subject_ids), "test", "train")

print(split_df["split"].value_counts())
print()
print("Event rate by split:")
print(split_df.groupby("split")["event_observed"].mean())
print()
print("Duration summary by split:")
print(split_df.groupby("split")["duration"].describe())

split
train    75380
test     19064
Name: count, dtype: int64

Event rate by split:
split
test     1.0
train    1.0
Name: event_observed, dtype: float64

Duration summary by split:
         count      mean       std       min       25%       50%       75%  \
split                                                                        
test   19064.0  3.679404  5.515839  0.003345  1.097506  1.975087  3.941976   
train  75380.0  3.617537  5.373389  0.001250  1.095943  1.962668  3.843134   

              max  
split              
test   121.303368  
train  226.403079  


In [17]:
readiness_report = {
    "row_count": len(modeling_df),
    "column_count": modeling_df.shape[1],
    "initial_predictor_count": len(predictor_cols),
    "ready_predictor_count": len(ready_predictor_cols),
    "dropped_high_missing_count": len(high_missing_cols),
    "dropped_low_variance_count": len(low_variance_cols),
    "numeric_predictor_count": len(ready_numeric_cols),
    "categorical_predictor_count": len(ready_categorical_cols),
    "event_observed_unique_values": modeling_df["event_observed"].nunique(),
    "duration_min": modeling_df["duration"].min(),
    "duration_median": modeling_df["duration"].median(),
    "duration_99th_pct": modeling_df["duration"].quantile(0.99),
    "duration_max": modeling_df["duration"].max(),
    "high_corr_pair_count_ge_0_98": len(high_corr_pairs_df),
}

readiness_report_df = pd.DataFrame(
    readiness_report.items(),
    columns=["check", "value"]
)

readiness_report_df

,check,value
0,row_count,94444.000000
1,column_count,457.000000
2,initial_predictor_count,448.000000
3,ready_predictor_count,443.000000
4,dropped_high_missing_count,5.000000
5,dropped_low_variance_count,0.000000
6,numeric_predictor_count,431.000000
7,categorical_predictor_count,12.000000
8,event_observed_unique_values,1.000000
9,duration_min,0.001250


In [18]:
pd.Series(ready_predictor_cols, name="predictor_column").to_csv(
    PROCESSED_DATA_DIR / "modeling_ready_predictor_columns.csv",
    index=False
)

pd.Series(ready_numeric_cols, name="numeric_predictor_column").to_csv(
    PROCESSED_DATA_DIR / "modeling_ready_numeric_columns.csv",
    index=False
)

pd.Series(ready_categorical_cols, name="categorical_predictor_column").to_csv(
    PROCESSED_DATA_DIR / "modeling_ready_categorical_columns.csv",
    index=False
)

pd.Series(drop_for_readiness, name="dropped_predictor_column").to_csv(
    PROCESSED_DATA_DIR / "modeling_dropped_predictor_columns.csv",
    index=False
)

missing_summary_df.to_csv(PROCESSED_DATA_DIR / "modeling_readiness_missingness.csv")
categorical_summary_df.to_csv(PROCESSED_DATA_DIR / "modeling_readiness_categorical_summary.csv", index=False)
high_corr_pairs_df.to_csv(PROCESSED_DATA_DIR / "modeling_high_correlation_pairs.csv", index=False)
split_df.to_csv(PROCESSED_DATA_DIR / "modeling_train_test_split.csv", index=False)
readiness_report_df.to_csv(PROCESSED_DATA_DIR / "modeling_readiness_report.csv", index=False)

print("Saved readiness artifacts to data/processed")

Saved readiness artifacts to data/processed


In [19]:
print("Notebook 10 readiness conclusion:")
print("- Core target and merge integrity checks passed.")
print("- Leakage columns are excluded from predictors.")
print("- Ready predictor lists and train/test split artifacts are saved.")
print("- event_observed is currently all 1, so model interpretation is time-to-ICU-discharge without censoring.")
print("- Cox proportional hazards assumptions should be checked after fitting a Cox model.")

Notebook 10 readiness conclusion:
- Core target and merge integrity checks passed.
- Leakage columns are excluded from predictors.
- Ready predictor lists and train/test split artifacts are saved.
- event_observed is currently all 1, so model interpretation is time-to-ICU-discharge without censoring.
- Cox proportional hazards assumptions should be checked after fitting a Cox model.
